<a href="https://colab.research.google.com/github/abdus-git/NLP/blob/main/unit3/32Hybrid_probabilistic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

class HybridBoundaryDetector:
    def __init__(self, transition_probs, emission_probs, state_rules):
        self.transition_probs = transition_probs
        self.emission_probs = emission_probs
        self.state_rules = state_rules
        self.states = ['FIRST', 'MIDDLE', 'LAST', 'SINGLE']

    def _get_probabilistic_score(self, page_features):
        scores = {}
        for state in self.states:
            prob = 1.0
            for feature, val in page_features.items():
                if feature in self.emission_probs[state]:
                    prob *= self.emission_probs[state][feature].get(val, 1e-6)
            scores[state] = prob
        return scores

    def predict_boundaries(self, sequence_features):
        predictions = []
        current_state = 'FIRST'

        for i, page_features in enumerate(sequence_features):
            prob_scores = self._get_probabilistic_score(page_features)
            allowed_transitions = self.state_rules.get(current_state, self.states)

            best_state = None
            best_score = -1.0

            for state in allowed_transitions:
                score = prob_scores[state] * self.transition_probs[current_state].get(state, 1e-6)
                if score > best_score:
                    best_score = score
                    best_state = state

            predictions.append(best_state)
            current_state = best_state

        return predictions

transitions = {
    'FIRST': {'MIDDLE': 0.7, 'LAST': 0.2, 'SINGLE': 0.1},
    'MIDDLE': {'MIDDLE': 0.5, 'LAST': 0.5},
    'LAST': {'FIRST': 0.8, 'SINGLE': 0.2},
    'SINGLE': {'FIRST': 0.8, 'SINGLE': 0.2}
}

emissions = {
    'FIRST': {'has_header': {True: 0.9, False: 0.1}, 'page_number': {'1': 0.95, 'other': 0.05}},
    'MIDDLE': {'has_header': {True: 0.2, False: 0.8}, 'page_number': {'1': 0.01, 'other': 0.99}},
    'LAST': {'has_header': {True: 0.1, False: 0.9}, 'page_number': {'1': 0.01, 'other': 0.99}},
    'SINGLE': {'has_header': {True: 0.9, False: 0.1}, 'page_number': {'1': 0.95, 'other': 0.05}}
}

fsm_rules = {
    'FIRST': ['MIDDLE', 'LAST', 'SINGLE'],
    'MIDDLE': ['MIDDLE', 'LAST'],
    'LAST': ['FIRST', 'SINGLE'],
    'SINGLE': ['FIRST', 'SINGLE']
}

detector = HybridBoundaryDetector(transitions, emissions, fsm_rules)

document_pages = [
    {'has_header': True, 'page_number': '1'},
    {'has_header': False, 'page_number': 'other'},
    {'has_header': False, 'page_number': 'other'},
    {'has_header': True, 'page_number': '1'}
]

print(detector.predict_boundaries(document_pages))

['SINGLE', 'FIRST', 'MIDDLE', 'MIDDLE']
